# 🙋🏻‍♂️🙋🏻‍♀️ <span style="color: white; background-color: SlateBlue"><b> Extração do Relatório de Candidaturas da Gupy </b></span></p>

🧩 <span style="color: CornflowerBlue"><b> 1- Acesso automatizado à Gupy </b></span></p>
- A automação abre via Selenium + PyAutoGUI
- A automação:
    - Abre o navegador via Selenium  
    - Acessa o portal da Gupy  
    - Preenche e envia login e senha  
    - Aguarda o carregamento completo da interface
- Esse bloco elimina totalmente a necessidade de acessar manualmente o portal para iniciar a extração.

📤 <span style="color: CornflowerBlue"><b> 2- Extração automática do relatório de candidaturas </b></span></p>
- Navega até a área de Relatórios  
- Seleciona o relatório application_report da sua conta  
- Preenche o período desejado:  'início fixo' (ex.: 01/12/2024) e 'fim' = ontem (dinâmico)
- Confirma a exportação  
- Aguarda a Gupy enviar o arquivo por e‑mail ao responsável
- Como a Gupy envia o relatório por e‑mail após o processamento, o script abre um prompt de confirmação para que o usuário valide a chegada do arquivo antes de continuar

📁 <span style="color: CornflowerBlue"><b> 3- Processamento dos arquivos baixados </b></span></p>
- O sistema procura, na pasta de Downloads, arquivos cujo nome começa com application_report_76663
- Para cada arquivo encontrado, executa:
    - Detecção automática do delimitador (vírgula ou ponto‑e‑vírgula)
    - Leitura robusta do CSV
    - Limpeza de caracteres ilegais para Excel
    - Padronização das colunas
    - Concatenação incremental
- Base consolidada é construída usando a chave:
    - ID da aplicação
    - Somente registros novos são adicionados

📦 <span style="color: CornflowerBlue"><b> 4- Organização e arquivamento dos arquivos </b></span></p>
- Cada arquivo processado é movido para 2. ARQUIVOS MOVIDOS
- O que garante histórico organizado e evita retrabalho
- O script também registra múltiplas etapas no PROCESSOS.xlsx:
    - Início  
    - Extração  
    - Processamento  
    - Geração  
    - Conclusão
    - Com timestamp e ID do processo

📊 <span style="color: CornflowerBlue"><b> 5- Geração do Excel final estruturado </b></span></p>
- O arquivo consolidado cria o arquivo CANDIDATURAS GUPY.xlsx
- Recebe:
    - Planilha única  
    - Tabela Excel (TableStyleLight13)  
    - Dados tratados, limpos e padronizados
    - Esse Excel é usado internamente para análises e integrações

🔃 <span style="color: CornflowerBlue"><b> 6- Atualização completa do Power BI </b></span></p>
- Abre o arquivo  CANDIDATURAS GUPY.pbix
- Aguarda o carregamento  
- Executa Atualizar  
- Aguarda concluir  
- Executa Publicar  
- Substitui o relatório existente no workspace  
- Fecha o Power BI
- Essa etapa sincroniza automaticamente o dashboard corporativo com a nova base de candidaturas

🗃️ <span style="color: CornflowerBlue"><b> 7- Exportação final para banco de dados </b></span></p>
- Criação do arquivo tb_gupy.csv
- Usado para ingestão em pipelines SQL, DW, ETL ou aplicações internas

⏱️ <span style="color: CornflowerBlue"><b> 8- Resumo e encerramento </b></span></p>
- Ao final, o script apresenta:
    - Mensagem de conclusão  
    - Tempo total de execução  
    - Logs detalhados no PROCESSOS.xlsx

# Importando as Bibliotecas

In [1]:
import csv
import datetime
import logging
import os
import pandas as pd
import pickle
import pyautogui
import pyperclip
import shutil
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime as dt
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
import openpyxl.utils.dataframe
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from playwright.sync_api import sync_playwright
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.datetime.today(), 0]
           
# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 5,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CANDIDATURAS GUPY.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'application_report_76663',
}

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Acessando a Gupy

In [4]:
load_dotenv(dotenv_path='env_path')

gupy_user = os.getenv("GUPY_USER")
gupy_password = os.getenv("GUPY_PASSWORD")
gupy_site = os.getenv("SITE_GUPY")

# Configuração de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

time.sleep(2)
pyautogui.hotkey('Ctrl', 't')

# Configuração de delays
pyautogui.PAUSE = 0.5

# Navegar até a URL
pyautogui.hotkey('ctrl', 'l')
time.sleep(0.5)
pyautogui.typewrite(gupy_site)
time.sleep(0.3)
pyautogui.press('enter')
time.sleep(10)  # Aguardar carregamento da página

# Aguardar username
pyautogui.click(x=259, y=489)
pyautogui.typewrite(gupy_user)
time.sleep(0.3)
pyautogui.press('enter')
time.sleep(2)

# Aguardar password
pyautogui.click(x=259, y=489)
pyautogui.hotkey('ctrl', 'a')
time.sleep(1)
pyautogui.press('del')
time.sleep(0.3)
pyautogui.typewrite(gupy_password)
time.sleep(0.3)
pyautogui.press('enter')

print("✅ Acessando a página da Gupy...")

✅ Acessando a página da Gupy...


# Baixando Relatório de Candidaturas

In [5]:
time.sleep(10)

# Home da Gupy
pyautogui.press("f5")
time.sleep(6)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(5)
pyautogui.click(x=737, y=397) # Candidaturas
time.sleep(2)
pyautogui.click(x=180, y=505) # CSV
time.sleep(0.5)
pyautogui.click(x=518, y=584) # Selecione um status de vaga
time.sleep(0.5)
pyautogui.click(x=232, y=617) # Selecionar todos
time.sleep(0.5)
pyautogui.click(x=1152, y=496) # Clicar fora
time.sleep(0.5)
pyautogui.click(x=967, y=581) # Selecione a vaga desejada
time.sleep(0.5)
pyautogui.click(x=972, y=627) # Selecionar todas as vagas
time.sleep(0.5)
pyautogui.click(x=1152, y=496) # Clicar fora

# Data atual = hoje - 1 dia
data_atual = datetime.date.today() - datetime.timedelta(days=1)

# Data início = começo da implatanção da Gupy
data_inicio_br = "01/12/2024"

# Formato brasileiro: DD/MM/AAAA
fmt_br = "%d/%m/%Y"

data_atual_br = data_atual.strftime(fmt_br)

time.sleep(1)
pyautogui.click(x=1028, y=707) # Clica em Data fim
time.sleep(0.5)
pyautogui.write(data_atual_br) # Digitar Data fim
time.sleep(0.5)
pyautogui.click(x=344, y=704) # Clica em Data início
time.sleep(0.5)
pyautogui.write(data_inicio_br) # Digitar Data início
time.sleep(0.5)
pyautogui.click(x=1340, y=790) # Clica em Enviar por e-mail
time.sleep(10)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Solicitação de download realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Solicitação de download realizado com sucesso

----------------------------------------------------------------------------------------------------


# Aguardando o Recebimento do E-mail da Gupy com o Arquivo para Download

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download das candidaturas?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-25 08:50:08,376 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-25 08:50:08,380 - INFO - Iniciando a leitura e processamento dos dados...


# Carga e Tratamento da Base 

In [7]:
# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# -------------------------
# Context Manager Workbook
# -------------------------
@contextmanager
def gerenciar_workbook(caminho: Path, sheet: str):
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

# -------------------------
# Carregar base principal
# -------------------------
def carregar_ou_criar_base(caminho: Path) -> pd.DataFrame:
    # Mudar extensão para .csv
    caminho_csv = caminho.with_suffix('.csv')
    
    if caminho_csv.exists():
        try:
            return pd.read_csv(caminho_csv, dtype=str)
        except Exception as e:
            logger.warning(f"⚠️ Arquivo CSV corrompido: {e}")
            caminho_csv.unlink()
            return pd.DataFrame()
    
    logger.warning(f"⚠️ {caminho_csv.name} não encontrado. Iniciando vazio.")
    return pd.DataFrame()

def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path, sheet_name: str = "CANDIDATURAS"):
    # Salvar CSV
    caminho_csv = caminho.with_suffix('.csv')
    logger.info(f"📝 Salvando CSV temporário...")
    df.to_csv(caminho_csv, index=False, encoding='utf-8')
    logger.info(f"✅ CSV salvo: {caminho_csv}")

    # 2. Criar Excel a partir do CSV
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    try:
        df.to_excel(caminho, sheet_name=sheet_name, index=False, engine="openpyxl")
        wb = load_workbook(caminho)
        ws = wb.active
        tabela = Table(displayName=sheet_name, ref=ws.dimensions)
        estilo = TableStyleInfo(
            name="TableStyleLight13",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=True,
        )
        tabela.tableStyleInfo = estilo
        ws.add_table(tabela)
        wb.save(caminho)
        logger.info(f"✅ Excel criado: {caminho}")
    except Exception as e:
        logger.error(f"❌ Erro ao criar Excel: {e}")
        logger.warning(f"⚠️ Usando apenas CSV como fallback")

# -------------------------
# Registrar etapa
# -------------------------
def registrar_etapa(caminho: Path, id_processo: int, etapa: int):
    with gerenciar_workbook(caminho, "REGISTROS") as ws:
        ws.append([id_processo, datetime.datetime.today(), etapa])
    logger.debug(f"Etapa {etapa} registrada")

# -------------------------
# Filtrar arquivos novos
# -------------------------
def filtrar_arquivos_novos(origem: Path, padrao: str = "application_report_76663") -> pd.DataFrame:
    arquivos = [f for f in origem.iterdir() if f.name.startswith(padrao)]

    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        return pd.DataFrame()

    df = pd.DataFrame([{"arquivo": f.name, "caminho": f} for f in arquivos])
    df["data"] = pd.to_datetime(df["arquivo"].str[25:39], format="%Y%m%d%H%M%S")
    df = df.sort_values(by="data", ascending=True)

    logger.info(f"📂 {len(df)} arquivo(s) encontrado(s)")
    return df

# -------------------------
# Consolidação incremental
# -------------------------
def consolidar_incremental(base_atual: pd.DataFrame, base_nova: pd.DataFrame, chave: str) -> pd.DataFrame:
    todas_colunas = base_atual.columns.union(base_nova.columns)
    base_atual = base_atual.reindex(columns=todas_colunas)
    base_nova = base_nova.reindex(columns=todas_colunas)

    novos = base_nova[~base_nova[chave].isin(base_atual[chave])]
    logger.debug(f"   +{len(novos)} novos registros")

    return (
        pd.concat([base_atual, novos], ignore_index=True)
        .drop_duplicates(subset=[chave], keep="first")
    )

# Realizar limpeza 
import re

XML_INVALID = re.compile(
    r'[^\x09\x0A\x0D\x20-\uD7FF'
    r'\uE000-\uFFFD'
    r'\U00010000-\U0010FFFF]'
)

def limpar_xml_safe(df):
    """Remove caracteres ilegais XML/Excel do DataFrame inteiro."""
    def clean(x):
        if isinstance(x, str):
            return XML_INVALID.sub('', x)
        return x
    return df.applymap(clean)

# -------------------------
# Leitura robusta do CSV
# -------------------------
def ler_csv_gupy(caminho):
    with open(caminho, 'r', encoding='utf-8', errors='replace') as f:
        amostra = f.read(4096)
        try:
            separador = csv.Sniffer().sniff(amostra).delimiter
        except:
            separador = ';'

    return pd.read_csv(
        caminho,
        sep=separador,
        encoding='utf-8',
        engine='python',
        dtype=str,
        on_bad_lines='skip'
    )

# =====================================================
# MAIN
# =====================================================
if __name__ == "__main__":

    logger.info("=" * 80)
    logger.info("🔄 INICIANDO PROCESSAMENTO DE CANDIDATURAS")
    logger.info("=" * 80 + "\n")

    try:
        # Etapa 0
        registrar_etapa(CONFIG["processos"], CONFIG["id_processo"], 0)

        # Carregar base atual
        logger.info("📋 Carregando base atual...")
        candidaturas_atual = carregar_ou_criar_base(CONFIG["saida"])
        logger.info(f"✅ {len(candidaturas_atual)} registros na base atual\n")

        registrar_etapa(CONFIG["processos"], CONFIG["id_processo"], 1)

        # Buscar arquivos
        logger.info("📂 Buscando arquivos novos...")
        df_arquivos = filtrar_arquivos_novos(CONFIG["origem"])

        if df_arquivos.empty:
            logger.warning("❌ Nenhum arquivo para processar")
            exit()

        registrar_etapa(CONFIG["processos"], CONFIG["id_processo"], 2)

        # Processar arquivos
        logger.info("🔄 Processando arquivos...\n")

        for idx, (_, row) in enumerate(df_arquivos.iterrows(), 1):

            nome_arquivo = row["arquivo"]
            caminho_arquivo = row["caminho"]

            logger.info(f"[{idx}/{len(df_arquivos)}] {nome_arquivo}")

            try:
                # Leitura CSV
                candidaturas_novas = ler_csv_gupy(caminho_arquivo)
                logger.debug(f"   ✓ {len(candidaturas_novas)} registros carregados")

                # Consolidar
                candidaturas_atual = consolidar_incremental(
                    candidaturas_atual,
                    candidaturas_novas,
                    chave="ID da aplicação"
                )

                # Mover arquivo
                destino = CONFIG["movidos"] / nome_arquivo
                shutil.move(str(caminho_arquivo), str(destino))
                logger.info(f"   ✓ Arquivo movido para: {destino}")

            except Exception as e:
                logger.error(f"   ❌ Erro ao processar {nome_arquivo}: {e}")
                continue

        registrar_etapa(CONFIG["processos"], CONFIG["id_processo"], 3)

        # Salvar resultado final
        logger.info("🧹 Limpando caracteres ilegais para Excel...")
        candidaturas_atual = limpar_xml_safe(candidaturas_atual)

        logger.info("💾 Salvando resultado final...")
        criar_excel_com_tabela(candidaturas_atual, CONFIG["saida"])

        registrar_etapa(CONFIG["processos"], CONFIG["id_processo"], 4)

        logger.info("\n" + "=" * 80)
        logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
        logger.info(f"   Total de registros: {len(candidaturas_atual)}")
        logger.info("=" * 80)

    except Exception as e:
        logger.error(f"\n❌ ERRO CRÍTICO: {e}")
        import traceback
        logger.error(traceback.format_exc())

2026-06-25 08:50:08,446 - INFO - ================================================================================
2026-06-25 08:50:08,458 - INFO - 🔄 INICIANDO PROCESSAMENTO DE CANDIDATURAS
2026-06-25 08:50:08,460 - INFO - ================================================================================

2026-06-25 08:50:08,875 - INFO - 📋 Carregando base atual...
2026-06-25 08:50:13,048 - INFO - ✅ 52073 registros na base atual

2026-06-25 08:50:13,559 - INFO - 📂 Buscando arquivos novos...
2026-06-25 08:50:13,581 - INFO - 📂 1 arquivo(s) encontrado(s)
2026-06-25 08:50:13,913 - INFO - 🔄 Processando arquivos...

2026-06-25 08:50:13,915 - INFO - [1/1] application_report_76663_20260625084728_t.csv
2026-06-25 08:50:29,095 - INFO -    ✓ Arquivo movido para: X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS\application_report_76663_20260625084728_t.csv
2026-06-25 08:50:29,498 - INFO - 🧹 Limpando caracteres ilegais para Excel...
C:\Users\rodrigo.bernandes\AppData\Local\Temp\ipykernel_2

# Atualização do Power BI - CANDIDATURAS GUPY

In [8]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\CANDIDATURAS GUPY.pbix"
os.startfile(path_pbi) # Abre o arquivo
time.sleep(45)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(5)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI atualizado

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [ ]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CANDIDATURAS GUPY.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_gupy.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

# Resumo de Finalização do Processo

In [ ]:
from datetime import datetime, date

tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')